# Decision model bakeoff

Compare Jev, Liquid D1, and local Laya on the same labeled scenarios.
Start with `MOCK = True` to exercise scoring and reports without services.
For real inference set it to `False` and select `MODEL_NAMES` below.
Jev needs TYPESAFE_API_KEY; D1 needs LIQUID_API_KEY. Laya downloads local
weights and uses HF_TOKEN when needed for Hugging Face access. Its hardware
requirements are the same as quickstart-laya.ipynb; no GPU is needed for mock runs.


In [1]:
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display
from system_one_models.decision_benchmark import create_backend, load_prompts, run_benchmark

MOCK = False
MODEL_NAMES = ["jev", "d1", "laya"]
OUTPUT_DIR = Path("/tmp/system-one-models-results")

# Match the credential locations used in the quickstart notebooks.
if not MOCK:
    for service in ("typesafe", "liquid", "huggingface"):
        load_dotenv(Path.home() / ".config" / service / "secrets.env")


## Create reusable model adapters

Each adapter supports `predict(state, questions)` and returns answers keyed by
question name. Hosted adapters accept `api_key`, `model`, and `base_url` overrides;
Laya accepts `token`, `preload`, and Router options. Clients initialize only here.
The quickstart-d1 notebook demonstrates hosted Liquid D1.


In [2]:
backends = []
try:
    for name in MODEL_NAMES:
        backends.append(create_backend(name, mock=MOCK))
except Exception:
    for backend in backends:
        if hasattr(backend, "close"):
            backend.close()
    raise
scenarios = load_prompts()
print(f"Loaded {len(scenarios)} scenarios, {sum(len(s['questions']) for s in scenarios)} questions")


/home/scott/github/system-one-models/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2754.34it/s]
/home/scott/github/system-one-models/.venv/lib/python3.13/site-packages/laya/router.py:774: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 533.80it/s]


Loaded 18 scenarios, 33 questions


## Run and inspect

Failures appear in the individual records and error counts. Request latency is
repeated for each question in that request. Score MAE uses the expected numeric
score; calibration uses the most probable level. Brier measures top-label
confidence versus correctness. Cost is unknown unless input-token pricing is
supplied; it excludes output-token and hardware costs.


In [3]:
try:
    result = run_benchmark(backends, scenarios)
    # Optionally pass input_prices_per_million={"jev": <current input price>}.
finally:
    for backend in backends:
        if hasattr(backend, "close"):
            backend.close()
display(Markdown(result.markdown()))


# Bake-off results

| backend | items | choice acc | noul acc | score MAE | score rho | ECE | Brier | p50 ms | est $/1k |
|---|---|---|---|---|---|---|---|---|---|
| jev | 33 | 0.923 | 0.857 | 0.370 | 0.883 | 0.073 | 0.089 | 119.9 | -- |
| d1 | 33 | 1.000 | 1.000 | 0.396 | 0.853 | 0.064 | 0.033 | 308.6 | -- |
| laya | 33 | 0.692 | 0.714 | 0.896 | 0.794 | 0.216 | 0.292 | 24.1 | -- |

_choice acc / noul acc_: fraction of exact-label matches. _score MAE_: mean |expected score - true level| (lower is better). _score rho_: Spearman rank correlation of predicted vs true levels. _ECE_: expected calibration error over probability bins (lower is better; values depend on the dataset). _Brier_: mean squared error of top-label probabilities versus correctness (lower is better). _est $/1k_: estimated USD per 1,000 scenario requests, using supplied input-token pricing._

## jev — reliability diagram
_33 scored items, 0 errors_

| prob bin | n | avg conf | emp acc |
|---|---|---|---|
| 0.0-0.5 | 0 | -- | -- |
| 0.5-0.6 | 2 | 0.580 | 0.000 |
| 0.6-0.7 | 2 | 0.640 | 0.500 |
| 0.7-0.8 | 1 | 0.740 | 1.000 |
| 0.8-0.9 | 7 | 0.841 | 0.857 |
| 0.9-1.0 | 21 | 0.981 | 0.952 |

## d1 — reliability diagram
_33 scored items, 0 errors_

| prob bin | n | avg conf | emp acc |
|---|---|---|---|
| 0.0-0.5 | 0 | -- | -- |
| 0.5-0.6 | 3 | 0.563 | 0.333 |
| 0.6-0.7 | 1 | 0.615 | 1.000 |
| 0.7-0.8 | 2 | 0.761 | 1.000 |
| 0.8-0.9 | 2 | 0.873 | 1.000 |
| 0.9-1.0 | 25 | 0.988 | 1.000 |

## laya — reliability diagram
_33 scored items, 0 errors_

| prob bin | n | avg conf | emp acc |
|---|---|---|---|
| 0.0-0.5 | 3 | 0.413 | 0.667 |
| 0.5-0.6 | 3 | 0.551 | 0.667 |
| 0.6-0.7 | 6 | 0.645 | 0.667 |
| 0.7-0.8 | 2 | 0.769 | 0.500 |
| 0.8-0.9 | 8 | 0.844 | 0.625 |
| 0.9-1.0 | 11 | 0.964 | 0.636 |


In [4]:
errors = [record for record in result.records if not record["ok"]]
errors[:5]


[]

In [5]:
result.write(OUTPUT_DIR)
print(f"Wrote results.md, results.csv, and results.json to {OUTPUT_DIR}")


Wrote results.md, results.csv, and results.json to /tmp/system-one-models-results
